# 🖼️ Multimodal RAG: Image & Text Embeddings (CLIP)

This notebook demonstrates how to process Shopify product images alongside their text descriptions using OpenAI's CLIP (Contrastive Language-Image Pre-Training) model. CLIP allows us to embed both text and images into the exact same vector space.

This means we can do three powerful things:
1. **Text-to-Image Search**: Search for "red floral dress" and find the dress image, even if the text description of the product never specifically says "red floral dress".
2. **Image-to-Image Search**: User uploads a photo of a jacket, we find visually similar jackets in the catalog.
3. **Image-to-Text Search**: We use the image to find relevant contextual text.

In [ ]:
import pandas as pd
import numpy as np
import faiss
from PIL import Image
import requests
from io import BytesIO
import time
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
# Ensure you have installed clip dependencies if needed: 
# pip install clip-retrieval
# pip install git+https://github.com/openai/CLIP.git

### 1. Load Data

We load the products that we prepared in Notebook 01, which includes an `image_url` column.

In [ ]:
df = pd.read_csv("shopify_products_prep.csv")
# Filter out products without images for this test
df = df[df["image_url"].notna()].reset_index(drop=True)
print(f"Loaded {len(df)} products with valid images.")

### 2. Helper to Download and Load Images

In [ ]:
def load_image_from_url(url):
    try:
        r = requests.get(url, timeout=5)
        img = Image.open(BytesIO(r.content)).convert("RGB")
        # Resize to save memory for batch processing
        img.thumbnail((300, 300))
        return img
    except Exception as e:
        print(f"Failed to load {url}: {e}")
        return None

### 3. Load Multimodal CLIP Model

We use `clip-ViT-B-32` from huggingface via the `sentence-transformers` library wrapper.

In [ ]:
print("Loading CLIP model...")
clip_model = SentenceTransformer('clip-ViT-B-32')
print("Model loaded!")

### 4. Create the Image & Text Vector Store

We will create **one** FAISS index containing image vectors.

In [ ]:
image_embeddings_list = []
valid_indices = []

print("Downloading and embedding images (this takes a while)...")
start_time = time.time()

# In production, you'd batch this and cache the images
for i, row in df.iterrows():
    img = load_image_from_url(row['image_url'])
    if img:
        # Get embedding for the image
        embed = clip_model.encode([img])[0]
        image_embeddings_list.append(embed)
        valid_indices.append(i)
        
    if i > 0 and i % 10 == 0:
        print(f"Processed {i} / {len(df)} images")

print(f"Embedded {len(image_embeddings_list)} images in {time.time() - start_time:.2f} seconds.")

# Filter dataset to only images we successfully embedded
df_clip = df.iloc[valid_indices].reset_index(drop=True)

image_embeddings = np.array(image_embeddings_list).astype("float32")
dimension = image_embeddings.shape[1]

print(f"Embedding dimension: {dimension}")

# Build FAISS Index
index_clip = faiss.IndexFlatL2(dimension)
index_clip.add(image_embeddings)
print(f"Index built with {index_clip.ntotal} vectors.")

### 5. Text-to-Image Search!
Since CLIP maps text and images to the same vector space, we can embed a pure-text string (like user search input) and query our FAISS index of *images* to find the closest match visually.

In [ ]:
def search_images_via_text(query, k=3):
    # Note we are embedding TEXT using the same CLIP model
    text_embed = clip_model.encode([query]).astype("float32")
    
    distances, indices = index_clip.search(text_embed, k)
    
    # Plot the results
    fig, axes = plt.subplots(1, k, figsize=(15, 5))
    fig.suptitle(f"Results for: '{query}'", fontsize=16)
    
    # Make axes iterable if k=1
    if k == 1: axes = [axes]
    
    for i, idx in enumerate(indices[0]):
        row = df_clip.iloc[idx]
        img = load_image_from_url(row['image_url'])
        
        ax = axes[i]
        if img:
            ax.imshow(img)
        ax.set_title(f"Score: {distances[0][i]:.2f}\n{row['title'][:30]}")
        ax.axis('off')
        
    plt.show()


# Try searching for visual traits that don't exist in your text descriptions!
search_images_via_text("A striped pattern top", k=3)
search_images_via_text("Something made of leather", k=3)
search_images_via_text("Vibrant red colors", k=3)

### 6. Image-to-Image Search

If a user uploads a photo to your chat (e.g. "I want shoes like this"), you embed the photo and find visually similar products.

In [ ]:
def search_similar_images(reference_img_path, k=3):
    query_img = Image.open(reference_img_path).convert("RGB")
    
    # Embed the specific uploaded image
    img_embed = clip_model.encode([query_img]).astype("float32")
    distances, indices = index_clip.search(img_embed, k)
    
    # ... you can plot these the exact same way as above ...
    return df_clip.iloc[indices[0]]

# (Requires a local image to test)
# similar_products = search_similar_images("test_jacket.jpg")